<a href="https://colab.research.google.com/github/nyhk-oi/agora-token-service/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@markdown # このノートブックについて

#@markdown このノートブックは[litagin02/Style-Bert-VITS2リポジトリにあるGoogle Colabノートブック](https://colab.research.google.com/github/litagin02/Style-Bert-VITS2/blob/master/colab.ipynb)をベースに、Google Driveに接続する機能を除き、声の学習を行なうためだけに簡略化させたノートブックとなります。

#@markdown プログラムが分かる方・より使いこなしていきたい方はベースとさせてもらった [litagin02/Style-Bert-VITS2](https://github.com/litagin02/Style-Bert-VITS2) をご覧ください。


In [ ]:
import os

if not os.path.exists("/content/Style-Bert-VITS2"):
    !git clone https://github.com/litagin02/Style-Bert-VITS2.git

%cd /content/Style-Bert-VITS2/

try:
    import loguru
    import numpy
    setup_done = (numpy.__version__ == "1.26.4")
except ImportError:
    setup_done = False

if not setup_done:
    print("初回セットアップを開始します...")

    # requirements-colab.txt で "==" により明示的にバージョン指定されている
    # パッケージ名を抽出(これらは制約ファイルから除外する)
    import re
    pinned = set()
    with open("requirements-colab.txt") as f:
        for line in f:
            line = line.strip()
            m = re.match(r"^([A-Za-z0-9_.\-]+)\s*==", line)
            if m:
                pinned.add(m.group(1).lower().replace("_", "-"))

    # 現在の環境の全パッケージを固定 (ただし上記の明示指定パッケージは除外)
    !pip freeze > /content/_freeze_all.txt
    with open("/content/_freeze_all.txt") as f:
        lines = f.readlines()

    with open("/content/_constraints.txt", "w") as f:
        for line in lines:
            name = re.split(r"==", line.strip())[0].lower().replace("_", "-")
            if name not in pinned:
                f.write(line)

    !pip install -r requirements-colab.txt -c /content/_constraints.txt --prefer-binary --timeout 120
    !pip uninstall -y numpy
    !pip install --no-cache-dir "numpy==1.26.4"
    !python initialize.py --skip_default_models
    print("環境を反映させるため再起動します。再起動後、もう一度このセルを実行してください。")
    import os; os.kill(os.getpid(), 9)
else:
    print("セットアップはすでに完了しています。")

dataset_root = "/content/dataset/Style-Bert-VITS2/Data"
assets_root = "/content/dataset/Style-Bert-VITS2/model_assets"
input_dir = "/content/dataset/Style-Bert-VITS2/inputs"

!mkdir -p {dataset_root}
!mkdir -p {assets_root}
!mkdir -p {input_dir}

print("準備が整いました。")

/content/Style-Bert-VITS2
初回セットアップを開始します...
  Using cached cmudict-1.1.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached cn2an-0.5.24-py3-none-any.whl.metadata (10 kB)
  Using cached g2p_en-2.1.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached librosa-0.9.2-py3-none-any.whl.metadata (8.2 kB)
  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
ERROR: Cannot install nltk<=3.8.1 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested nltk<=3.8.1
    The user requested (constraint) nltk==3.9.1

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict

ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
     ━━━━━━━━━

Traceback (most recent call last):
  File "/content/Style-Bert-VITS2/initialize.py", line 9, in <module>
    from style_bert_vits2.logging import logger
  File "/content/Style-Bert-VITS2/style_bert_vits2/logging.py", line 1, in <module>
    from loguru import logger
ModuleNotFoundError: No module named 'loguru'


In [2]:
import os, re

%cd /content/Style-Bert-VITS2

# Colab の constraint から nltk / numpy だけ除いたものを作る。
# 全部外すと pip が pandas までバックトラックして sdist ビルドに失敗する。
orig = os.environ.get('PIP_CONSTRAINT', '')
custom = '/content/constraints-sbv2.txt'

if orig and os.path.exists(orig):
    with open(orig) as f:
        lines = f.readlines()
    kept = [l for l in lines
            if not re.match(r'^\s*(nltk|numpy)\s*[=<>!]', l, re.I)]
    with open(custom, 'w') as f:
        f.writelines(kept)
    print(f'constraint: {orig} -> {custom} ({len(lines)} -> {len(kept)} lines)')
else:
    open(custom, 'w').close()
    print('no original constraint found; using empty one')

os.environ['PIP_CONSTRAINT'] = custom

# pyannote.audio 4.x は numpy>=2 要求で requirements の numpy<2 と衝突する。3 系に固定。
!sed -i 's/^nltk.*/nltk/' requirements-colab.txt
!sed -i 's/^pyannote\.audio.*/pyannote.audio==3.3.2/' requirements-colab.txt
!cat requirements-colab.txt

!pip install -r requirements-colab.txt
!pip install numpy==1.26.4

import importlib.util
for m in ['loguru', 'librosa', 'cmudict', 'g2p_en', 'pyannote.audio', 'pyopenjtalk']:
    print(m, 'OK' if importlib.util.find_spec(m) else 'MISSING')

/content/Style-Bert-VITS2
no original constraint found; using empty one
accelerate
cmudict
cn2an
g2p_en
gradio>=4.32
jieba
librosa==0.9.2
loguru
nltk
num2words
numpy<2
onnx
onnxconverter-common
onnxruntime
onnxruntime-gpu
onnxsim-prebuilt
pyannote.audio==3.3.2
pyloudnorm
pyopenjtalk-dict
pypinyin
pyworld-prebuilt
torch
torchaudio
torchvision
transformers
umap-learn
  Using cached cmudict-1.1.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached cn2an-0.5.24-py3-none-any.whl.metadata (10 kB)
  Using cached g2p_en-2.1.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached librosa-0.9.2-py3-none-any.whl.metadata (8.2 kB)
  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
  Using cached num2words-0.5.14-py3-none-any.whl.metadata (13 kB)
  Using cached onnx-1.22.0-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.5 kB)
  Using cached onnxconverter_common-1.16.0-py2.py3-none-any.whl.metadata (4.8 kB)
  Using cached onnxruntime-1.29.0-cp312-cp312-manylinux_2_28_x86_

In [3]:
%cd /content/Style-Bert-VITS2

import numpy
print('numpy', numpy.__version__)   # 1.26.4 であること

!python initialize.py

/content/Style-Bert-VITS2
numpy 1.26.4
08-18 14:56:49 |  INFO  | initialize.py:19 | Downloading deberta-v2-large-japanese-char-wwm pytorch_model.bin

pytorch_model.bin: downloading bytes:  15% 197M/1.32G [00:01<00:04, 256MB/s, 14.3MB/s  ]
pytorch_model.bin: downloading bytes:  23% 309M/1.32G [00:01<00:03, 262MB/s, 26.6MB/s  ]
pytorch_model.bin: downloading bytes:  93% 1.22G/1.32G [00:04<00:00, 362MB/s, 97.9MB/s  ]
pytorch_model.bin: reconstructing file:  46% 604M/1.32G [00:06<00:07, 91.7MB/s, 25.4MB/s  ]
pytorch_model.bin: downloading bytes: 100% 1.25G/1.25G [00:15<00:00, 80.9MB/s,  101MB/s  ]
pytorch_model.bin: reconstructing file: 100% 1.32G/1.32G [00:15<00:00, 85.0MB/s, 77.3MB/s  ]
08-18 14:57:05 |  INFO  | initialize.py:19 | Downloading deberta-v2-large-japanese-char-wwm-onnx model_fp16.onnx

model_fp16.onnx: downloading bytes:   8% 53.9M/653M [00:00<00:06, 86.7MB/s, 2.47MB/s  ]
model_fp16.onnx: downloading bytes:  14% 91.1M/653M [00:01<00:03, 146MB/s, 5.26MB/s  ] 
model_fp16.onnx:

In [6]:
import site, os

sitecustomize_code = '''
import torchaudio
if not hasattr(torchaudio, "list_audio_backends"):
    def list_audio_backends():
        return ["soundfile"]
    torchaudio.list_audio_backends = list_audio_backends
'''

sitecustomize_path = os.path.join(site.getsitepackages()[0], "sitecustomize.py")
with open(sitecustomize_path, "w", encoding="utf-8") as f:
    f.write(sitecustomize_code)

print(f"パッチを書き込みました: {sitecustomize_path}")

# 動作確認
import subprocess
result = subprocess.run(
    ["python", "-c", "import torchaudio; print(torchaudio.list_audio_backends())"],
    capture_output=True, text=True
)
print(result.stdout, result.stderr)

パッチを書き込みました: /usr/local/lib/python3.12/dist-packages/sitecustomize.py
 Traceback (most recent call last):
  File "<string>", line 1, in <module>
AttributeError: module 'torchaudio' has no attribute 'list_audio_backends'



In [8]:
!grep -rn "list_audio_backends" /root/.cache/torch/hub/litagin02_silero-vad_master/ 2>/dev/null
!grep -rn "list_audio_backends" /content/Style-Bert-VITS2/ 2>/dev/null
!python -c "import torchaudio; print(hasattr(torchaudio, 'list_audio_backends')); print(torchaudio.list_audio_backends())"

/root/.cache/torch/hub/litagin02_silero-vad_master/utils_vad.py:127:    audio_backends = torchaudio.list_audio_backends()
False
Traceback (most recent call last):
  File "<string>", line 1, in <module>
AttributeError: module 'torchaudio' has no attribute 'list_audio_backends'


In [9]:
target_file = "/root/.cache/torch/hub/litagin02_silero-vad_master/utils_vad.py"

with open(target_file, "r", encoding="utf-8") as f:
    content = f.read()

old_line = "audio_backends = torchaudio.list_audio_backends()"
new_line = 'audio_backends = torchaudio.list_audio_backends() if hasattr(torchaudio, "list_audio_backends") else ["soundfile"]'

if old_line in content:
    content = content.replace(old_line, new_line)
    with open(target_file, "w", encoding="utf-8") as f:
        f.write(content)
    print("パッチ適用完了")
else:
    print("対象の行が見つかりませんでした。ファイルの内容を確認してください。")

パッチ適用完了


In [10]:
!grep -n "audio_backends" /root/.cache/torch/hub/litagin02_silero-vad_master/utils_vad.py

127:    audio_backends = torchaudio.list_audio_backends() if hasattr(torchaudio, "list_audio_backends") else ["soundfile"]
129:    if len(sox_backends.intersection(audio_backends)) > 0:


In [12]:
!ffprobe "/content/dataset/Style-Bert-VITS2/inputs/Fujippi_Serif_1 (2).wav" 2>&1 | head -30

ffprobe version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2007-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --ena

In [13]:
import os, glob, subprocess

input_dir = "/content/dataset/Style-Bert-VITS2/inputs"
converted_dir = "/content/converted_inputs"
os.makedirs(converted_dir, exist_ok=True)

wav_files = glob.glob(os.path.join(input_dir, "*.wav"))
print(f"{len(wav_files)} 件のファイルを変換します")

for f in wav_files:
    basename = os.path.basename(f)
    out_path = os.path.join(converted_dir, basename)
    cmd = [
        "ffmpeg", "-y", "-i", f,
        "-ar", "44100",      # サンプリングレートを44100Hzに統一
        "-ac", "1",          # モノラルに統一
        "-c:a", "pcm_s16le", # 16bit PCMに変換
        out_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"失敗: {basename}")
        print(result.stderr[-500:])
    else:
        print(f"成功: {basename}")

3 件のファイルを変換します
成功: Fujippi_Serif_2 (2).wav
成功: Fujippi_Serif_3 (2).wav
成功: Fujippi_Serif_1 (2).wav


In [14]:
import shutil

# 元のファイルを削除して、変換済みファイルに入れ替える
for f in glob.glob(os.path.join(input_dir, "*.wav")):
    os.remove(f)

for f in glob.glob(os.path.join(converted_dir, "*.wav")):
    shutil.copy(f, input_dir)

print("入れ替え完了")
!ls -la {input_dir}

入れ替え完了
total 7296
drwxr-xr-x 2 root root    4096 Aug 18 15:27  .
drwxr-xr-x 5 root root    4096 Aug 18 14:48  ..
-rw-r--r-- 1 root root 2554918 Aug 18 15:27 'Fujippi_Serif_1 (2).wav'
-rw-r--r-- 1 root root 2270506 Aug 18 15:27 'Fujippi_Serif_2 (2).wav'
-rw-r--r-- 1 root root 2632308 Aug 18 15:27 'Fujippi_Serif_3 (2).wav'


In [16]:
!grep -n "AudioMetaData" /usr/local/lib/python3.12/dist-packages/pyannote/audio/core/io.py

60:) -> torchaudio.AudioMetaData:
75:    info : torchaudio.AudioMetaData


In [18]:
import re

target_file = "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/io.py"

with open(target_file, "r", encoding="utf-8") as f:
    content = f.read()

patch_code = '''
if not hasattr(torchaudio, "AudioMetaData"):
    class _AudioMetaDataShim:
        def __init__(self, sample_rate=0, num_frames=0, num_channels=0, bits_per_sample=0, encoding=""):
            self.sample_rate = sample_rate
            self.num_frames = num_frames
            self.num_channels = num_channels
            self.bits_per_sample = bits_per_sample
            self.encoding = encoding
    torchaudio.AudioMetaData = _AudioMetaDataShim
'''

lines = content.split("\n")
new_lines = []
inserted = False
for line in lines:
    new_lines.append(line)
    if not inserted and re.match(r"^import torchaudio\s*$", line):
        new_lines.append(patch_code)
        inserted = True

if inserted:
    with open(target_file, "w", encoding="utf-8") as f:
        f.write("\n".join(new_lines))
    print("パッチ適用完了")
else:
    print("import torchaudio の行が見つかりませんでした。冒頭を確認します。")
    print("\n".join(lines[:20]))

パッチ適用完了


In [19]:
!python -c "from pyannote.audio import Inference, Model; print('OK')"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/__init__.py", line 29, in <module>
    from .core.inference import Inference
  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/inference.py", line 36, in <module>
    from pyannote.audio.core.model import Model, Specifications
  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/model.py", line 47, in <module>
    from pyannote.audio.core.task import (
  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/task.py", line 51, in <module>
    from pyannote.audio.utils.protocol import check_protocol
  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/protocol.py", line 31, in <module>
    get_duration = Audio(mono="downmix").get_duration
                   ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/io.py", line 223, in __init__
    torchaudio.lis

In [20]:
import torchaudio
torchaudio_init_path = torchaudio.__file__
print(torchaudio_init_path)

/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py


In [21]:
patch_code = '''

# --- 互換性パッチ: 新しいtorchaudioで削除されたAPIをダミーで復元 ---
if not hasattr(torchaudio, "list_audio_backends"):
    def list_audio_backends():
        return ["soundfile"]
    torchaudio.list_audio_backends = list_audio_backends

if not hasattr(torchaudio, "AudioMetaData"):
    class _AudioMetaDataShim:
        def __init__(self, sample_rate=0, num_frames=0, num_channels=0, bits_per_sample=0, encoding=""):
            self.sample_rate = sample_rate
            self.num_frames = num_frames
            self.num_channels = num_channels
            self.bits_per_sample = bits_per_sample
            self.encoding = encoding
    torchaudio.AudioMetaData = _AudioMetaDataShim
# --- パッチここまで ---
'''

target_file = "/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py"

with open(target_file, "r", encoding="utf-8") as f:
    content = f.read()

if "互換性パッチ" not in content:
    with open(target_file, "a", encoding="utf-8") as f:
        f.write(patch_code)
    print(f"パッチ適用完了: {target_file}")
else:
    print("すでにパッチ適用済みです")

パッチ適用完了: /usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py


In [22]:
!python -c "import torchaudio; print(torchaudio.list_audio_backends()); print(torchaudio.AudioMetaData)"
!python -c "from pyannote.audio import Inference, Model; print('OK')"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py", line 208, in <module>
    if not hasattr(torchaudio, "list_audio_backends"):
                   ^^^^^^^^^^
NameError: name 'torchaudio' is not defined
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/__init__.py", line 29, in <module>
    from .core.inference import Inference
  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/inference.py", line 35, in <module>
    from pyannote.audio.core.io import AudioFile
  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/io.py", line 39, in <module>
    import torchaudio
  File "/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py", line 208, in <module>
    if not hasattr(torchaudio, "list_audio_backends"):
                   ^^^^^^^^^^
NameError: name 'torchaudio' is not 

In [23]:
target_file = "/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py"

with open(target_file, "r", encoding="utf-8") as f:
    content = f.read()

marker_start = "\n\n# --- 互換性パッチ"
idx = content.find(marker_start)
if idx != -1:
    content = content[:idx]
    with open(target_file, "w", encoding="utf-8") as f:
        f.write(content)
    print("以前のパッチを削除しました")
else:
    print("パッチが見つかりませんでした(既に削除済みかも)")

以前のパッチを削除しました


In [24]:
patch_code = '''

# --- 互換性パッチ: 新しいtorchaudioで削除されたAPIをダミーで復元 ---
if "list_audio_backends" not in globals():
    def list_audio_backends():
        return ["soundfile"]

if "AudioMetaData" not in globals():
    class AudioMetaData:
        def __init__(self, sample_rate=0, num_frames=0, num_channels=0, bits_per_sample=0, encoding=""):
            self.sample_rate = sample_rate
            self.num_frames = num_frames
            self.num_channels = num_channels
            self.bits_per_sample = bits_per_sample
            self.encoding = encoding
# --- パッチここまで ---
'''

with open(target_file, "r", encoding="utf-8") as f:
    content = f.read()

if "互換性パッチ" not in content:
    with open(target_file, "a", encoding="utf-8") as f:
        f.write(patch_code)
    print("パッチ適用完了")
else:
    print("すでにパッチ適用済みです")

パッチ適用完了


In [25]:
!python -c "import torchaudio; print(torchaudio.list_audio_backends()); print(torchaudio.AudioMetaData)"
!python -c "from pyannote.audio import Inference, Model; print('OK')"

['soundfile']
<class 'torchaudio.AudioMetaData'>
OK


In [27]:
target_file = None
import huggingface_hub
print(huggingface_hub.__file__)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py


In [28]:
patch_code = '''

# --- 互換性パッチ: use_auth_token -> token の引数名変換 ---
from huggingface_hub.file_download import hf_hub_download as _original_hf_hub_download

def _patched_hf_hub_download(*args, **kwargs):
    if "use_auth_token" in kwargs:
        kwargs["token"] = kwargs.pop("use_auth_token")
    return _original_hf_hub_download(*args, **kwargs)

import huggingface_hub.file_download as _hf_file_download_module
_hf_file_download_module.hf_hub_download = _patched_hf_hub_download

hf_hub_download = _patched_hf_hub_download
# --- パッチここまで ---
'''

init_file = "/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py"

with open(init_file, "r", encoding="utf-8") as f:
    content = f.read()

if "互換性パッチ" not in content:
    with open(init_file, "a", encoding="utf-8") as f:
        f.write(patch_code)
    print(f"パッチ適用完了: {init_file}")
else:
    print("すでにパッチ適用済みです")

パッチ適用完了: /usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py


In [32]:
with open("/tmp/check1.py", "w", encoding="utf-8") as f:
    f.write("""
from huggingface_hub import hf_hub_download
import inspect
print(inspect.signature(hf_hub_download))
""")

!python /tmp/check1.py

(*args, **kwargs)


In [33]:
with open("/tmp/check2.py", "w", encoding="utf-8") as f:
    f.write("""
from pyannote.audio import Model
model = Model.from_pretrained('pyannote/wespeaker-voxceleb-resnet34-LM', use_auth_token=None)
print('OK')
""")

!python /tmp/check2.py


pytorch_model.bin: downloading bytes:  76% 20.2M/26.6M [00:00<00:00, 36.9MB/s,  421kB/s  ]
pytorch_model.bin: downloading bytes: 100% 25.2M/25.2M [00:00<00:00, 33.5MB/s, 2.47MB/s  ]
pytorch_model.bin: reconstructing file: 100% 26.6M/26.6M [00:00<00:00, 35.4MB/s, 2.63MB/s  ]
config.yaml: 100% 221/221 [00:00<00:00, 1.45MB/s]
Traceback (most recent call last):
  File "/tmp/check2.py", line 3, in <module>
    model = Model.from_pretrained('pyannote/wespeaker-voxceleb-resnet34-LM', use_auth_token=None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/model.py", line 671, in from_pretrained
    loaded_checkpoint = pl_load(path_for_pl, map_location=map_location)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py", line 73, in _load
    return torch.load(
           ^^^^^

In [34]:
with open("/tmp/check3.py", "w", encoding="utf-8") as f:
    f.write("""
import torch
import torch.torch_version
torch.serialization.add_safe_globals([torch.torch_version.TorchVersion])

from pyannote.audio import Model
model = Model.from_pretrained('pyannote/wespeaker-voxceleb-resnet34-LM', use_auth_token=None)
print('OK')
""")

!python /tmp/check3.py

Traceback (most recent call last):
  File "/tmp/check3.py", line 7, in <module>
    model = Model.from_pretrained('pyannote/wespeaker-voxceleb-resnet34-LM', use_auth_token=None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/model.py", line 671, in from_pretrained
    loaded_checkpoint = pl_load(path_for_pl, map_location=map_location)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py", line 73, in _load
    return torch.load(
           ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1578, in load
    raise pickle.UnpicklingError(_get_wo_message(str(e))) from None
_pickle.UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of

In [35]:
patch_code = '''

# --- 互換性パッチ: torch.load のデフォルトを weights_only=False に戻す ---
# (信頼できる公式モデル読み込みのため。PyTorch 2.6以降のデフォルト変更に対応)
_original_torch_load = torch.load

def _patched_torch_load(*args, **kwargs):
    if "weights_only" not in kwargs:
        kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)

torch.load = _patched_torch_load
# --- パッチここまで ---
'''

target_file = "/usr/local/lib/python3.12/dist-packages/torch/__init__.py"

with open(target_file, "r", encoding="utf-8") as f:
    content = f.read()

if "torch.load のデフォルトを weights_only=False" not in content:
    with open(target_file, "a", encoding="utf-8") as f:
        f.write(patch_code)
    print(f"パッチ適用完了: {target_file}")
else:
    print("すでにパッチ適用済みです")

パッチ適用完了: /usr/local/lib/python3.12/dist-packages/torch/__init__.py


In [36]:
patch_code = '''

# --- 互換性パッチ: pyannote.audioのチェックポイント読み込みのためsafe globalsを許可 ---
try:
    import torch.torch_version
    torch.serialization.add_safe_globals([torch.torch_version.TorchVersion])
except Exception:
    pass
# --- パッチここまで ---
'''

import torch
target_file = torch.__file__

with open(target_file, "r", encoding="utf-8") as f:
    content = f.read()

if "互換性パッチ" not in content:
    with open(target_file, "a", encoding="utf-8") as f:
        f.write(patch_code)
    print(f"パッチ適用完了: {target_file}")
else:
    print("すでにパッチ適用済みです")

すでにパッチ適用済みです


In [37]:
!python /tmp/check3.py

Traceback (most recent call last):
  File "/tmp/check3.py", line 7, in <module>
    model = Model.from_pretrained('pyannote/wespeaker-voxceleb-resnet34-LM', use_auth_token=None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/model.py", line 643, in from_pretrained
    _ = hf_hub_download(
        ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py", line 1813, in _patched_hf_hub_download
    return _original_hf_hub_download(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 88, in _inner_fn
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py", line 1013, in hf_hub_download
  File "/usr/local/lib/python3.12/dist-packages/huggingfa

In [39]:
target_file = "/usr/local/lib/python3.12/dist-packages/torch/__init__.py"

with open(target_file, "r", encoding="utf-8") as f:
    content = f.read()

old_patch = '''def _patched_torch_load(*args, **kwargs):
    if "weights_only" not in kwargs:
        kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)'''

new_patch = '''def _patched_torch_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)'''

if old_patch in content:
    content = content.replace(old_patch, new_patch)
    with open(target_file, "w", encoding="utf-8") as f:
        f.write(content)
    print("パッチを修正しました(常に weights_only=False に強制)")
else:
    print("該当箇所が見つかりませんでした。ファイル末尾を確認します。")
    print(content[-800:])

パッチを修正しました(常に weights_only=False に強制)


In [40]:
### 一度ランタイムを再起動するため、必要な情報を再度セットアップ
# 改めて作業ディレクトリに移動
%cd /content/Style-Bert-VITS2

# 学習に必要なファイルや途中経過が保存されるディレクトリ
dataset_root = "/content/dataset/Style-Bert-VITS2/Data"

# 学習結果（音声合成に必要なファイルたち）が保存されるディレクトリ
assets_root = "/content/dataset/Style-Bert-VITS2/model_assets"

# 元となる音声ファイル（wav形式）を入れるディレクトリ
input_dir = "/content/dataset/Style-Bert-VITS2/inputs"

import yaml

with open("configs/paths.yml", "w", encoding="utf-8") as f:
    yaml.dump({"dataset_root": dataset_root, "assets_root": assets_root}, f)

#@markdown ---
#@markdown # 作成するモデル関連の内容を入力
#@markdown ※デフォルトのままでも問題ありません
# モデル名（話者名）を入力
model_name = "fujipi221" #@param {type:"string"}

# こういうふうに書き起こして欲しいという例文（句読点の入れ方・笑い方や固有名詞等）
initial_prompt = "こんにちは。元気、ですかー？ふふっ、私は……ちゃんと元気だよ！" #@param {type:"string"}

!python slice.py -i {input_dir} --model_name {model_name}
!python transcribe.py --model_name {model_name} --initial_prompt {initial_prompt} --use_hf_whisper --hf_repo_id openai/whisper-large-v3 --language ja

#@markdown ---
#@markdown ## JP-Extraを有効化する
#@markdown JP-Extra （日本語特化版）を有効化すると、日本語の能力が向上する代わりに英語と中国語は使えなくなります。
use_jp_extra = True #@param {type: "boolean"}

#@markdown ---
#@markdown ## 学習のバッチサイズ。
#@markdown VRAMのはみ出具合に応じて調整してください。
batch_size = 4 #@param {type:"string"}

#@markdown ---
#@markdown ## 学習のエポック数
# 100で多すぎるほどかもしれませんが、もっと多くやると質が上がる可能性もあります
epochs = 100 #@param {type:"string"}

#@markdown ---
#@markdown ## 保存頻度
#@markdown 保存頻度。何ステップごとにモデルを保存するか。分からなければデフォルトのままで。
save_every_steps = 1000 #@param {type:"string"}

#@markdown ---
#@markdown 音声ファイルの音量を正規化するかどうか
normalize = False #@param {type:"string"}

#@markdown ---
#@markdown 音声ファイルの開始・終了にある無音区間を削除するかどうか
trim = False #@param {type:"string"}


#@markdown ---
#@markdown 読みのエラーが出た場合にどうするか。

#@markdown ・"raise"ならテキスト前処理が終わったら中断

#@markdown ・"skip"なら読めない行は学習に使わない

#@markdown ・"use"なら無理やり使う
yomi_error = "skip" #@param {type:"string"}

# 以降は学習に関する処理
from gradio_tabs.train import preprocess_all
from style_bert_vits2.nlp.japanese import pyopenjtalk_worker
import yaml
from gradio_tabs.train import get_path

pyopenjtalk_worker.initialize_worker()

preprocess_all(
    model_name=model_name,
    batch_size=batch_size,
    epochs=epochs,
    save_every_steps=save_every_steps,
    num_processes=2,
    normalize=normalize,
    trim=trim,
    freeze_EN_bert=False,
    freeze_JP_bert=False,
    freeze_ZH_bert=False,
    freeze_style=False,
    freeze_decoder=False,
    use_jp_extra=use_jp_extra,
    val_per_lang=0,
    log_interval=200,
    yomi_error=yomi_error,
)

paths = get_path(model_name)
dataset_path = str(paths.dataset_path)
config_path = str(paths.config_path)

with open("default_config.yml", "r", encoding="utf-8") as f:
    yml_data = yaml.safe_load(f)
yml_data["model_name"] = model_name
with open("config.yml", "w", encoding="utf-8") as f:
    yaml.dump(yml_data, f, allow_unicode=True)

if use_jp_extra:
  # 学習 （日本語特化版を「使う」場合）
  !python train_ms_jp_extra.py --config {config_path} --model {dataset_path} --assets_root {assets_root}
else:
  # 学習 （日本語特化版を「使わない」場合）
  !python train_ms.py --config {config_path} --model {dataset_path} --assets_root {assets_root}

/content/Style-Bert-VITS2
08-18 15:47:46 |  INFO  | slice.py:167 | Found 3 audio files.
08-18 15:47:46 |WARNING | slice.py:169 | Output directory /content/dataset/Style-Bert-VITS2/Data/fujipi221/raw already exists, deleting...
Using cache found in /root/.cache/torch/hub/litagin02_silero-vad_master
  0%|          | 0/3 [00:00<?, ?it/s]Using cache found in /root/.cache/torch/hub/litagin02_silero-vad_master
Using cache found in /root/.cache/torch/hub/litagin02_silero-vad_master
Using cache found in /root/.cache/torch/hub/litagin02_silero-vad_master
100%|##########| 3/3 [00:02<00:00,  1.04it/s]
08-18 15:47:50 |  INFO  | slice.py:265 | Slice done! Total time: 1.20 min, 13 files.
08-18 15:47:53 |  INFO  | transcribe.py:157 | Found 13 WAV files
08-18 15:47:53 |WARNING | transcribe.py:163 | /content/dataset/Style-Bert-VITS2/Data/fujipi221/esd.list exists, backing up to /content/dataset/Style-Bert-VITS2/Data/fujipi221/esd.list.bak
08-18 15:47:53 |WARNING | transcribe.py:166 | /content/dataset/S

In [42]:
!grep -n "from_pretrained\|torch_dtype\|dtype" /content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py

98:            DebertaV2Model.from_pretrained(
106:        __loaded_models[language] = AutoModelForMaskedLM.from_pretrained(
166:        __loaded_tokenizers[language] = DebertaV2TokenizerFast.from_pretrained(
172:        __loaded_tokenizers[language] = AutoTokenizer.from_pretrained(


In [43]:
!sed -n '90,115p' /content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py

        pretrained_model_name_or_path = str(DEFAULT_BERT_MODEL_PATHS[language])

    # BERT モデルをロードし、辞書に格納して返す
    ## 英語のみ DebertaV2Model でロードする必要がある
    start_time = time.time()
    if language == Languages.EN:
        __loaded_models[language] = cast(
            DebertaV2Model,
            DebertaV2Model.from_pretrained(
                pretrained_model_name_or_path,
                device_map=device_map,
                cache_dir=cache_dir,
                revision=revision,
            ),
        )
    else:
        __loaded_models[language] = AutoModelForMaskedLM.from_pretrained(
            pretrained_model_name_or_path,
            device_map=device_map,
            cache_dir=cache_dir,
            revision=revision,
        )
    logger.info(
        f"Loaded the {language.name} BERT model from {pretrained_model_name_or_path} ({time.time() - start_time:.2f}s)"
    )



In [44]:
target_file = "/content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py"

with open(target_file, "r", encoding="utf-8") as f:
    content = f.read()

old_block = '''    if language == Languages.EN:
        __loaded_models[language] = cast(
            DebertaV2Model,
            DebertaV2Model.from_pretrained(
                pretrained_model_name_or_path,
                device_map=device_map,
                cache_dir=cache_dir,
                revision=revision,
            ),
        )
    else:
        __loaded_models[language] = AutoModelForMaskedLM.from_pretrained(
            pretrained_model_name_or_path,
            device_map=device_map,
            cache_dir=cache_dir,
            revision=revision,
        )'''

new_block = '''    if language == Languages.EN:
        __loaded_models[language] = cast(
            DebertaV2Model,
            DebertaV2Model.from_pretrained(
                pretrained_model_name_or_path,
                device_map=device_map,
                cache_dir=cache_dir,
                revision=revision,
                dtype=torch.float32,
            ),
        )
    else:
        __loaded_models[language] = AutoModelForMaskedLM.from_pretrained(
            pretrained_model_name_or_path,
            device_map=device_map,
            cache_dir=cache_dir,
            revision=revision,
            dtype=torch.float32,
        )'''

if old_block in content:
    content = content.replace(old_block, new_block)
    with open(target_file, "w", encoding="utf-8") as f:
        f.write(content)
    print("パッチ適用完了")
else:
    print("該当箇所が見つかりませんでした。ファイルの内容が想定と異なります。")

パッチ適用完了


In [45]:
!grep -n "^import torch\|^import torch " /content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py

In [47]:
target_file = "/content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py"

with open(target_file, "r", encoding="utf-8") as f:
    content = f.read()

if "\nimport torch\n" not in content and not content.startswith("import torch\n"):
    # 最初のimport文の前に追加する
    lines = content.split("\n")
    insert_idx = 0
    for i, line in enumerate(lines):
        if line.startswith("import ") or line.startswith("from "):
            insert_idx = i
            break
    lines.insert(insert_idx, "import torch")
    content = "\n".join(lines)
    with open(target_file, "w", encoding="utf-8") as f:
        f.write(content)
    print("import torch を追加しました")
else:
    print("すでに import torch があります")

import torch を追加しました


In [48]:
!grep -n "^import torch" /content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py

11:import torch


In [51]:
!sed -n '1,15p' /content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py

"""
Style-Bert-VITS2 の学習・推論に必要な各言語ごとの BERT モデルをロード/取得するためのモジュール。

オリジナルの Bert-VITS2 では各言語ごとの BERT モデルが初回インポート時にハードコードされたパスから「暗黙的に」ロードされているが、
場合によっては多重にロードされて非効率なほか、BERT モデルのロード元のパスがハードコードされているためライブラリ化ができない。

そこで、ライブラリの利用前に、音声合成に利用する言語の BERT モデルだけを「明示的に」ロードできるようにした。
一度 load_model/tokenizer() で当該言語の BERT モデルがロードされていれば、ライブラリ内部のどこからでもロード済みのモデル/トークナイザーを取得できる。
"""

import torch
from __future__ import annotations

import gc
import time


In [52]:
target_file = "/content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py"

with open(target_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

# 誤って追加された "import torch\n" 行を削除(複数あれば全部)
lines = [l for l in lines if l.strip() != "import torch"]

# from __future__ import annotations の直後に import torch を挿入
new_lines = []
inserted = False
for line in lines:
    new_lines.append(line)
    if not inserted and line.strip() == "from __future__ import annotations":
        new_lines.append("import torch\n")
        inserted = True

if not inserted:
    new_lines.insert(0, "import torch\n")

with open(target_file, "w", encoding="utf-8") as f:
    f.writelines(new_lines)

print("修正完了")

修正完了


In [53]:
!sed -n '1,20p' /content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py

"""
Style-Bert-VITS2 の学習・推論に必要な各言語ごとの BERT モデルをロード/取得するためのモジュール。

オリジナルの Bert-VITS2 では各言語ごとの BERT モデルが初回インポート時にハードコードされたパスから「暗黙的に」ロードされているが、
場合によっては多重にロードされて非効率なほか、BERT モデルのロード元のパスがハードコードされているためライブラリ化ができない。

そこで、ライブラリの利用前に、音声合成に利用する言語の BERT モデルだけを「明示的に」ロードできるようにした。
一度 load_model/tokenizer() で当該言語の BERT モデルがロードされていれば、ライブラリ内部のどこからでもロード済みのモデル/トークナイザーを取得できる。
"""

from __future__ import annotations
import torch

import gc
import time
from typing import TYPE_CHECKING, Optional, Union, cast

from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,


In [55]:
!sed -n '25,45p' /content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py

    PreTrainedTokenizerFast,
)

from style_bert_vits2.constants import DEFAULT_BERT_MODEL_PATHS, Languages
from style_bert_vits2.logging import logger
from style_bert_vits2.nlp import onnx_bert_models


if TYPE_CHECKING:


# 各言語ごとのロード済みの BERT モデルを格納する辞書
__loaded_models: dict[Languages, Union[PreTrainedModel, DebertaV2Model]] = {}

# 各言語ごとのロード済みの BERT トークナイザーを格納する辞書
__loaded_tokenizers: dict[
    Languages,
    Union[PreTrainedTokenizer, PreTrainedTokenizerFast, DebertaV2TokenizerFast],
] = {}




In [56]:
!curl -s https://raw.githubusercontent.com/litagin02/Style-Bert-VITS2/master/style_bert_vits2/nlp/bert_models.py -o /tmp/bert_models_original.py
!sed -n '1,45p' /tmp/bert_models_original.py


"""
Style-Bert-VITS2 の学習・推論に必要な各言語ごとの BERT モデルをロード/取得するためのモジュール。

オリジナルの Bert-VITS2 では各言語ごとの BERT モデルが初回インポート時にハードコードされたパスから「暗黙的に」ロードされているが、
場合によっては多重にロードされて非効率なほか、BERT モデルのロード元のパスがハードコードされているためライブラリ化ができない。

そこで、ライブラリの利用前に、音声合成に利用する言語の BERT モデルだけを「明示的に」ロードできるようにした。
一度 load_model/tokenizer() で当該言語の BERT モデルがロードされていれば、ライブラリ内部のどこからでもロード済みのモデル/トークナイザーを取得できる。
"""

from __future__ import annotations

import gc
import time
from typing import TYPE_CHECKING, Optional, Union, cast

from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    DebertaV2Model,
    DebertaV2TokenizerFast,
    PreTrainedModel,
    PreTrainedTokenizer,
    PreTrainedTokenizerFast,
)

from style_bert_vits2.constants import DEFAULT_BERT_MODEL_PATHS, Languages
from style_bert_vits2.logging import logger
from style_bert_vits2.nlp import onnx_bert_models


if TYPE_CHECKING:
    import torch


# 各言語ごとのロード済みの BERT モデルを格納する辞書
__loaded_models: dict[Languages, Union[PreTrainedModel, DebertaV2Model]] = {}

# 各言語ごと

In [57]:
target_file = "/content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py"

with open(target_file, "r", encoding="utf-8") as f:
    content = f.read()

old_block = """from style_bert_vits2.nlp import onnx_bert_models


if TYPE_CHECKING:


# 各言語ごとのロード済みの BERT モデルを格納する辞書"""

new_block = """from style_bert_vits2.nlp import onnx_bert_models

import torch

if TYPE_CHECKING:
    import torch


# 各言語ごとのロード済みの BERT モデルを格納する辞書"""

if old_block in content:
    content = content.replace(old_block, new_block)
    with open(target_file, "w", encoding="utf-8") as f:
        f.write(content)
    print("修正完了")
else:
    print("該当ブロックが見つかりませんでした")

修正完了


In [58]:
!sed -n '1,40p' /content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py

"""
Style-Bert-VITS2 の学習・推論に必要な各言語ごとの BERT モデルをロード/取得するためのモジュール。

オリジナルの Bert-VITS2 では各言語ごとの BERT モデルが初回インポート時にハードコードされたパスから「暗黙的に」ロードされているが、
場合によっては多重にロードされて非効率なほか、BERT モデルのロード元のパスがハードコードされているためライブラリ化ができない。

そこで、ライブラリの利用前に、音声合成に利用する言語の BERT モデルだけを「明示的に」ロードできるようにした。
一度 load_model/tokenizer() で当該言語の BERT モデルがロードされていれば、ライブラリ内部のどこからでもロード済みのモデル/トークナイザーを取得できる。
"""

from __future__ import annotations
import torch

import gc
import time
from typing import TYPE_CHECKING, Optional, Union, cast

from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    DebertaV2Model,
    DebertaV2TokenizerFast,
    PreTrainedModel,
    PreTrainedTokenizer,
    PreTrainedTokenizerFast,
)

from style_bert_vits2.constants import DEFAULT_BERT_MODEL_PATHS, Languages
from style_bert_vits2.logging import logger
from style_bert_vits2.nlp import onnx_bert_models

import torch

if TYPE_CHECKING:
    import torch


# 各言語ごとのロード済みの BERT モデルを格納する辞書
__loaded_models: dict[Languages, Union[PreTrainedModel, Debe

In [59]:
!grep -n "dtype=torch.float32" /content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py


105:                dtype=torch.float32,
114:            dtype=torch.float32,


In [ ]:
#@markdown # Web UI起動
#@markdown Web UIを起動して学習させたモデルを試す事ができますが、Colabからだと音声の生成ができないかも？
!python app.py --share

08-18 16:18:51 | DEBUG  | __init__.py:130 | pyopenjtalk worker server started
2026-08-18 16:19:00.877607: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/content/Style-Bert-VITS2/app.py:52: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=GRADIO_THEME) as app:
/content/Style-Bert-VITS2/gradio_tabs/inference.py:364: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=GRADIO_THEME) as app:
/content/Style-Bert-VITS2/gradio_tabs/dataset.py:118: UserWarning: The parameters hav

In [ ]:
#@markdown # ONNX変換
#@markdown 学習済みのモデルをONMXに変換します。

#@markdown 変換後は

#@markdown /content/dataset/Style-Bert-VITS2/model_assets/{設定したモデル名}/{設定したモデル名}_e100_s300.onnx

#@markdown にモデルが出力されます
!git fetch -p
!git checkout -b dev origin/dev
!pip install -r requirements-colab.txt
!python convert_onnx.py --model /content/dataset/Style-Bert-VITS2/model_assets/your_model_name